# 02 — Preparação dos dados
**Hackathon Databricks · Terra Signal · Grupo 4**

**Equipe:** André Ferreira, Antonio Menezes, Júlia Vale e Lucas Menghi

**O que este notebook faz:** lê as tabelas bronze, aplica as decisões de limpeza tomadas pelo grupo em 10/11 e grava
as tabelas limpas (`silver_history` e `silver_inference`), que o notebook 04 usa para montar a gold. Também converte os tipos: números como números e
`Churn` como 1 (cancelou) e 0 (ficou).

## Checklist de decisões

**É a mesma tabela em todos os notebooks.** As linhas com ☑ foram decididas pelo grupo em 10/11, com as respostas do
Carlos (Databricks). Se uma decisão mudar, a função `preparar` muda junto.

| # | Decisão | Opções | Proposta | Impacto | Evidência no 01 | No 02 | Depende de | Decisão do grupo |
|---|---|---|---|---|---|---|---|---|
| D1 | Grafia de `PhoneService` (70 + 23) | Padronizar | Padronizar para `Yes`/`No` | Sem isso, `yes` e `Yes` viram categorias diferentes no modelo | 4.1 | Aplicado | Carlos: só erro de padronização (respondido) | ☑ Padronizar para `Yes`/`No` (10/11) |
| D2 | `TotalCharges` nulo (278 + 64) | a) preencher com `MonthlyCharges` × `tenure`; b) excluir; c) deixar ausente | a | Excluir não serve para os 64 da inferência, que precisam de previsão, e perde o sinal da 4.3. A regra vale para 99% dos clientes | 4.1, 4.2, 4.3 | Aplicado | G2; Carlos: preencher assim é "razoável, desde que lembrem que é uma aproximação" (respondido); falta saber por que o valor está ausente | ☑ Preencher com `MonthlyCharges` × `tenure`, como aproximação (10/11) |
| D3 | `TotalCharges` nulo com `tenure` = 0 (10 + 1) | a) 0; b) outro valor | a (sai da própria regra: mensalidade × 0) | Clientes novos, sem cobrança ainda; efeito pequeno | 4.2 | Aplicado | G2 | ☑ 0 (10/11) |
| D4 | `tenure` = `unknown` (19 + 4) | a) `TotalCharges` ÷ `MonthlyCharges`, arredondado; b) mediana; c) excluir | a; mediana do histórico no único cliente sem as duas colunas | `tenure` é das variáveis mais ligadas ao churn: um erro aqui muda a faixa de tempo de casa do cliente | 4.1, 4.2 | Aplicado | G2; Carlos: 72 meses é teto real (respondido) | ☑ `TotalCharges` ÷ `MonthlyCharges`, arredondado e limitado a 72; mediana do histórico no caso sem os dois (10/11) |
| D5 | Indicador `fl_total_charges_ausente` | Criar ou não | Criar e comparar o modelo com e sem ele | Preserva o sinal da ausência (33,1% contra 26,4% de churn, p = 0,016) | 4.3 | Criado | Carlos: por que falta `TotalCharges` | ☑ Criar; o modelo testa com e sem (10/11) |
| D6 | Indicador `fl_tenure_ausente` | Criar ou não | Criar, mas tende a sair na seleção (só 19 casos, p = 0,21) | Pouca informação; uma coluna a mais | 4.3 | Não criado | G2 | ☑ Não criar: 19 casos, p = 0,21 (10/11) |
| D7 | `TotalCharges` fora de ±20% da regra (47 + 9) | a) manter; b) investigar; c) excluir | a | Mostra que o preenchimento de D2 é uma aproximação | 4.2 | Mantidos | Carlos: o preço muda com plano e promoções, e não há histórico de preços (respondido) | ☑ Manter (10/11) |
| D8 | Quebras de linha no `CustomerFeedback` (85 + 15) | Remover ou manter | Remover, se o texto for usado | Só importa se o texto entrar na análise ou no modelo | 4.2 | Aplicado | D9 | ☑ Trocar a quebra de linha por espaço (10/11) |
| D9 | `CustomerFeedback` no modelo | a) com; b) sem; c) os dois | c, com o modelo sem texto como principal | Possível vazamento: uma regra sobre o texto acerta 98,9% | Célula 2 (seção 1); 7.3 do 03 | Fora do 02 (modelo) | G1; Carlos: feedback antes ou depois do cancelamento | ☑ Fora da ABT. O texto vai na tabela separada `gold_churn_texto`, para o modelo com texto (contraexemplo); o modelo oficial é sem texto (10/11) |
| D10 | Serviços com 3 categorias ou sim/não | a) manter 3; b) binário | Testar as duas | "No internet service" repete `InternetService` em 6 colunas: redundância | 3 | Fora do 02 (notebook 04) | G4 | ◐ Sugestão: serviços como sim/não (`_bin`), sempre junto com `InternetService`. A ABT traz as duas versões; seleção final na modelagem (10/11) |
| D11 | Fonte e schema do grupo | a) CSVs de `dados/`; b) tabelas `bronze_*` em `terra_signal` | b | Todos os notebooks; os números precisam bater entre os integrantes | 1.2, 2 | Aplicado: o 02 lê das tabelas `bronze_*` | G3; Carlos: CSV com quebra de linha, schema | ☑ Tabelas `bronze_*` em `workspace.terra_signal`; ignorar (sem apagar) as do upload em `workspace.default` (10/11) |
| D12 | `MonthlyIncome` no modelo | a) manter como candidata; b) ignorar | a, e deixar a seleção decidir | Base externa, sem garantia de precisão; os organizadores dizem que pode ser ignorada. Na exploratória, a renda não muda o risco | 3; seção 10 do 03 | Convertida para número e mantida | Carlos: renda familiar em R$, base externa (respondido) | ◐ Sugestão: descartar (IV abaixo de 0,02, base externa). Fica na ABT; seleção final na modelagem (10/11) |
| D13 | Linhas do `Prediction.csv` | a) 1.409 (clientes); b) 1.461 (linhas físicas) | a | Arquivo com o número errado de linhas pode invalidar a entrega | 1.2 | Fora do 02 (entrega) | Organização: confirmar | ☑ 1.409, uma por `customerID`; a confirmar com a organização (10/11) |
| D14 | Formato da gold | — | `gold_churn_treino` (5.634, com `Churn` 1/0) e `gold_churn_inferencia` (1.409, sem `Churn`), mesmas colunas e ordem; números como número, categorias como texto; comentário em cada coluna; `gold_churn_previsoes` depois do modelo | É o que a modelagem usa; formato estável evita retrabalho | — | Fora do 02 (notebook 04) | — | ☑ ABT com todas as candidatas (originais limpas, serviços nas duas versões, indicador e derivadas) + dicionário com a sugestão de cada coluna; texto em `gold_churn_texto` (10/11) |
| D15 | Outras variáveis sem poder preditivo | a) descartar já; b) deixar a seleção decidir | a: `gender`, `PhoneService` e `MultipleLines` | IV abaixo de 0,02; modelo mais simples de explicar; sexo é variável sensível | Seção 7 e 8 do 03 | Fora do 02 (notebook 04) | — | ◐ Sugestão: descartar `gender`, `PhoneService` e `MultipleLines` (IV abaixo de 0,02). Ficam na ABT; seleção final na modelagem (10/11) |
| D16 | `MonthlyCharges` e `TotalCharges` (repetem outras variáveis) | a) manter; b) trocar por derivada; c) deixar a seleção decidir | Decidir no 04, com as derivadas | Repetem `InternetService` e `tenure` (associação ≥ 0,8) | Seção 6 e 8 do 03 | Fora do 02 (notebook 04) | 04 | ◐ Sugestão: manter `MonthlyCharges` (ou a mensalidade relativa ao pacote) e descartar `TotalCharges` (repete `tenure`, ρ 0,89). As três ficam na ABT; seleção final na modelagem (10/11) |
| D17 | Fonte do 03 | a) CSVs; b) tabelas `bronze_*` | b | Coerência com a D11; os números não mudam | — | Fora do 02 (notebook 03) | — | ☑ Ler da bronze (10/11) |
| D18 | Divisão de papéis | — | A preparação entrega a ABT e sugere; a seleção final das variáveis é feita na modelagem | Quem decide a seleção final das variáveis | — | — | — | ☑ Como proposto (10/11) |
| D19 | Seleção de variáveis | a) só as recomendadas; b) recomendadas + seleção para frente por AUC; c) testes estatísticos com correção para múltiplos testes e retirada das duplicatas | c, conferida com b | É o que entra no modelo | — | — | Notebook 05 | ☑ As 41 colunas do notebook 05 (seções 2 e 3): saem 6 categóricas redundantes e 12 variáveis (duplicatas exatas, versões repetidas do tempo de casa, `gender` sem associação após a correção FDR). A conferência com os métodos da aula (11 variáveis, AUC 0,841) cabe dentro das 41 (11/11) |
| D20 | Segmentação de clientes | a) sem segmentação; b) K-means no comportamento, sem o churn, cruzado com risco × valor | b | Diz que tipo de cliente está em risco e que oferta faz sentido | — | — | Notebook 07 | ◐ Sugestão para o negócio e a apresentação (11/11) |
| D21 | Modelo final | a) logística simples; b) regularização (Lasso, Ridge, Elastic Net); c) árvores (*random forest*, XGBoost, CatBoost) | O de melhor desempenho e mais consistente entre treino, teste e base total | É o modelo que vai para a entrega | — | — | Notebook 06 | ☑ Notebook 06 (modelo do grupo): logística Lasso, 41 colunas, C = 0,3, classes balanceadas, vencedora pela regra de consistência (Ridge a 0,003); AUC 0,830 no teste. Atenção: o balanceamento infla a probabilidade (média de 41% para 27% de churn real) (11/11) |
| D22 | Ponto de corte do Sim/Não | a) 0,5; b) o que maximiza o F1; c) pelo custo da ação | b | Define quantos clientes viram "Yes" | — | — | Notebook 06 | ☑ 0,5046: maior F1 nas probabilidades fora da dobra do treino (notebook 06) (11/11) |
| D23 | Arquivo da entrega | — | 1.409 linhas, `customerID` e `prediction` (Yes/No), na ordem do `inference.csv`, no formato do notebook oficial | Arquivo fora do formato pode invalidar a entrega | — | — | D13; notebook 08 | ☑ `prediction_grupo4_<data_hora>.csv` gerado no 08 e conferido nas 8 regras: 571 de 1.409 com Yes (40,5%); falta comparar os nomes das colunas com o notebook oficial (11/11) |
| D24 | Monitoramento do modelo | a) sem monitoramento; b) PSI das variáveis e das notas, taxa de Yes e reprodução pelo MLflow, com semáforo e app | b | Avisa quando o modelo envelhece, antes de o resultado de negócio mostrar | — | — | Notebook 09 | ◐ Proposta: notebook 09 e app em Streamlit (11/11) |
| D25 | Motor de atendimento | a) só a fila de retenção; b) a nota, os motivos e a oferta de cada cliente na hora da ligação, e a comparação de pacotes para quem ainda não é cliente | b | Leva o modelo para quem fala com o cliente | — | — | Notebook 10 | ◐ Proposta: notebook 10 e página Atendimento do app (12/11) |

**Como ler:** "Evidência no 01" aponta a seção do notebook 01. "No 02" diz se o notebook 02 já aplica a proposta.
"Depende de" liga às perguntas para o grupo (G1 a G4) e ao Carlos. ☑ = decidido (com a data); ◐ = sugestão nossa, a confirmar (as de variáveis, na modelagem); ☐ = em aberto.

In [ ]:
%run ./_comum

## 1. Ler a bronze (D11)

A fonte única do grupo são as tabelas `bronze_*` gravadas no notebook 01, e não os CSVs. As colunas de controle
(`_origem`, `_carregado_em`) saem aqui, porque não são variáveis do negócio.

In [ ]:
bronze_hist = spark.table(f"{FQ}.bronze_history").toPandas().drop(columns=["_origem", "_carregado_em"])
bronze_inf = spark.table(f"{FQ}.bronze_inference").toPandas().drop(columns=["_origem", "_carregado_em"])
assert (len(bronze_hist), len(bronze_inf)) == (5634, 1409), "a bronze não tem o número de clientes esperado: rodar o notebook 01"
print(f"bronze_history: {virg(len(bronze_hist), 0)} clientes · bronze_inference: {virg(len(bronze_inf), 0)} clientes")

## 2. Corrigir

In [ ]:
MEDIANA_TENURE = pd.to_numeric(bronze_hist["tenure"], errors="coerce").median()   # do histórico (treino), também na inferência

def preparar(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    tenure = pd.to_numeric(df["tenure"], errors="coerce")
    mensal = pd.to_numeric(df["MonthlyCharges"], errors="coerce")
    total = pd.to_numeric(df["TotalCharges"], errors="coerce")

    df["fl_total_charges_ausente"] = total.isna().astype(int)                  # D5: marca antes de corrigir (D6: tenure não)

    recuperado = (total / mensal).round().clip(upper=72)                     # D4: total ÷ mensalidade, no máximo 72 (teto real)
    tenure = tenure.fillna(recuperado).fillna(MEDIANA_TENURE)                # D4: senão, mediana
    total = total.fillna(mensal * tenure)                                     # D2 e D3: mensalidade × meses (0 se tenure = 0)

    df["tenure"] = tenure.astype(int)
    df["MonthlyCharges"] = mensal
    df["TotalCharges"] = total.round(2)
    df["MonthlyIncome"] = pd.to_numeric(df["MonthlyIncome"], errors="coerce")
    df["SeniorCitizen"] = df["SeniorCitizen"].astype(int)
    df["PhoneService"] = df["PhoneService"].str.capitalize()                  # D1: yes/no -> Yes/No
    df["CustomerFeedback"] = df["CustomerFeedback"].str.replace(r"\s*\n\s*", " ", regex=True)   # D8: quebra de linha -> espaço
    if "Churn" in df.columns:
        df["Churn"] = df["Churn"].eq("Yes").astype(int)
    return df

hist_limpo = preparar(bronze_hist)
inf_limpo = preparar(bronze_inf)

## 3. Conferir o antes e depois

In [ ]:
def contar_problemas(df: pd.DataFrame) -> dict:
    return {
        "tenure inválido": int(pd.to_numeric(df["tenure"], errors="coerce").isna().sum()),
        "tenure acima do teto de 72 meses": int((pd.to_numeric(df["tenure"], errors="coerce") > 72).sum()),
        "TotalCharges inválido": int(pd.to_numeric(df["TotalCharges"], errors="coerce").isna().sum()),
        "PhoneService em minúsculo": int(df["PhoneService"].isin(["yes", "no"]).sum()),
        "CustomerFeedback com quebra de linha": int(df["CustomerFeedback"].str.contains("\n").sum()),
    }

conferencia = pd.DataFrame({"historico_antes": contar_problemas(bronze_hist), "historico_depois": contar_problemas(hist_limpo),
                            "inferencia_antes": contar_problemas(bronze_inf), "inferencia_depois": contar_problemas(inf_limpo)})
display(conferencia.reset_index(names="problema"))
assert conferencia[["historico_depois", "inferencia_depois"]].sum().sum() == 0, "sobrou problema depois da limpeza"
assert len(hist_limpo) == len(bronze_hist) and len(inf_limpo) == len(bronze_inf), "o número de clientes mudou"
print(f"Taxa de churn no histórico: {virg(100 * hist_limpo['Churn'].mean(), 1)}%")
print(f"Clientes com TotalCharges = 0 (tenure = 0): {int((hist_limpo['TotalCharges'] == 0).sum())} no histórico, "
      f"{int((inf_limpo['TotalCharges'] == 0).sum())} na inferência")
display(hist_limpo.dtypes.astype(str).rename("tipo").rename_axis("coluna").reset_index())

**O que vimos:** nenhum problema restante, o número de clientes não mudou, e os tipos estão certos: numéricas como
números, `Churn` como 1/0 e o indicador `fl_total_charges_ausente` criado.

## 4. Gravar as tabelas limpas

In [ ]:
for nome, df in [("silver_history", hist_limpo), ("silver_inference", inf_limpo)]:
    (spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.{nome}"))
    print(f"{FQ}.{nome}: {spark.table(f'{FQ}.{nome}').count()} linhas")

## Resumo

| O que fiz | Resultado |
|---|---|
| Li a fonte única do grupo (D11) | `bronze_history` e `bronze_inference`, sem as colunas de controle |
| Corrigi três colunas | `tenure` (19 + 4, D4), `TotalCharges` (278 + 64, D2 e D3) e `PhoneService` (70 + 23, D1) |
| Limpei o texto | Quebras de linha do `CustomerFeedback` trocadas por espaço (85 + 15, D8) |
| Criei um indicador de ausência | `fl_total_charges_ausente` (D5). O de `tenure` não foi criado (D6) |
| Gravei as tabelas limpas | `workspace.terra_signal.silver_history` e `silver_inference` |
| Checklist | D1 a D8 e D11 decididos e aplicados em 10/11; D9, D10 e D12 ficam para o 03 e o 04 |

**Próximo passo:** notebook 03, análise exploratória.

## Checklist final do 02

O que este notebook aplicou e conferiu, e o que fica para os próximos. As linhas seguem o checklist do início.

| # | Decisão | Status no 02 | Conferido em |
|---|---|---|---|
| D11 | Ler das tabelas `bronze_*` | ☑ Aplicado | Seção 1: 5.634 e 1.409 clientes |
| D1 | `PhoneService` → `Yes`/`No` | ☑ Aplicado | Seção 3: 70 + 23 → 0 |
| D2 | `TotalCharges` nulo = `MonthlyCharges` × `tenure` | ☑ Aplicado | Seção 3: 278 + 64 → 0 |
| D3 | `TotalCharges` = 0 com `tenure` = 0 | ☑ Aplicado | Seção 3: 10 + 1 clientes com total 0 |
| D4 | `tenure` = `TotalCharges` ÷ `MonthlyCharges`, limitado a 72; mediana no caso sem os dois | ☑ Aplicado | Seção 3: 19 + 4 → 0; nenhum acima de 72 |
| D5 | Criar `fl_total_charges_ausente` | ☑ Aplicado | Seção 3: coluna presente, tipo inteiro |
| D6 | Não criar `fl_tenure_ausente` | ☑ Aplicado | Seção 3: coluna ausente |
| D7 | Manter os 47 + 9 fora de ±20% | ☑ Aplicado | Nenhuma linha excluída: 5.634 e 1.409 clientes |
| D8 | Quebra de linha do feedback → espaço | ☑ Aplicado | Seção 3: 85 + 15 → 0 |
| D13 | `Prediction.csv` com 1.409 linhas | ☑ Decidido (entrega) | Silver da inferência com 1.409 linhas |
| D9 | `CustomerFeedback` no modelo | ☑ Decidido no 03: fora da ABT e do modelo oficial | — |
| D10 | Serviços com 3 categorias ou sim/não | ◐ Sugestão no 03; as duas versões na ABT (04) | — |
| D12 | `MonthlyIncome` no modelo | ◐ Sugestão no 03: descartar; fica na ABT | — |
| D14 | Formato da gold (ABT) | ☑ Decidido | Aplicado no 04 |
| D15 | `gender`, `PhoneService`, `MultipleLines` | ◐ Sugestão no 03: descartar; ficam na ABT | — |
| D16 | `MonthlyCharges` e `TotalCharges` | ◐ Sugestão no 04 | — |
| D17 | Fonte do 03 | ☑ Decidido | Aplicado no 03 |
| D18 | Divisão de papéis: a preparação entrega a ABT e sugere | ☑ Decidido | — |
| D19 | Seleção de variáveis | ☑ As 41 colunas do 05 (testes com correção FDR e retirada das duplicatas); a conferência com os métodos da aula cabe dentro delas (11/11) | Notebook 05 |
| D20 | Segmentação de clientes | ◐ K-means no comportamento (5 segmentos), cruzado com risco × valor (11/11) | Notebook 07 |
| D21 | Modelo final | ☑ Logística Lasso, 41 colunas, C = 0,3, classes balanceadas; AUC 0,830 no teste (11/11) | Notebook 06 |
| D22 | Ponto de corte do Sim/Não | ☑ 0,5046, maior F1 fora da dobra do treino (11/11) | Notebook 06 |
| D23 | Arquivo da entrega | ☑ Gerado e conferido nas 8 regras: 571 de 1.409 com Yes (11/11) | Notebook 08 |

**Saída do 02:** `silver_history` (5.634) e `silver_inference` (1.409), 24 colunas, churn de 26,7% (igual ao bruto).